In [1]:
import sys, os
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from sklearn.feature_selection import mutual_info_classif
from sklearn.utils.class_weight import compute_class_weight



src_path = os.path.abspath("../src")
if src_path not in sys.path:
    sys.path.append(src_path)

%load_ext autoreload
%autoreload 2

SAMPLE_DIR = r"D:\projects\Healthcare\data\raw_sample"


from data_cleaning import clean_dataset
from feature_engineering import process_features
from sklearn.model_selection import train_test_split

df_raw = pd.read_csv(os.path.join(SAMPLE_DIR, "stroke_sliced.csv"))
df_clean, _ = clean_dataset(df_raw)
df_processed, _ = process_features(
    df_clean, target_col="Diagnosis",
    artifacts_path="../models/preprocessing_artifacts.pkl"
)

print("Pipeline ready. Processed shape:", df_processed.shape)

2026-07-03 16:45:44,895 | data_cleaning | INFO | Cleaning started. Input shape: (15000, 22)
2026-07-03 16:45:45,007 | data_cleaning | INFO | Filled 'Symptoms' with mode (16.7% missing)
2026-07-03 16:45:45,139 | data_cleaning | INFO | Removed 0 duplicate rows
2026-07-03 16:45:45,191 | data_cleaning | INFO | Skipped 'Hypertension' for outliers (2 unique values)
2026-07-03 16:45:45,197 | data_cleaning | INFO | Skipped 'Heart Disease' for outliers (2 unique values)
2026-07-03 16:45:45,229 | data_cleaning | INFO | Skipped 'Stroke History' for outliers (2 unique values)
2026-07-03 16:45:45,243 | data_cleaning | INFO | Cleaning finished. Output shape: (15000, 22)
2026-07-03 16:45:45,245 | feature_engineering | INFO | Feature processing started. Input shape: (15000, 22)
2026-07-03 16:45:50,633 | feature_engineering | INFO | Feature engineering: dropped 5 columns
2026-07-03 16:45:50,825 | feature_engineering | INFO | Encoding: 3 binary, 6 one-hot
2026-07-03 16:45:50,875 | feature_engineering | 

Pipeline ready. Processed shape: (15000, 35)


In [2]:
from sklearn.model_selection import train_test_split

def split_data(df, target_col, test_size=0.2, random_state=42):
    
    X = df.drop(columns=[target_col])
    y = df[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        stratify=y,
        random_state=random_state,
    )
    return X_train, X_test, y_train, y_test



from data_cleaning import clean_dataset
from feature_engineering import process_features

df_raw = pd.read_csv(os.path.join(SAMPLE_DIR, "stroke_sliced.csv"))
df_clean, _ = clean_dataset(df_raw)
df_processed, _ = process_features(df_clean, target_col="Diagnosis",
                                   artifacts_path="../models/preprocessing_artifacts.pkl")


X_train, X_test, y_train, y_test = split_data(df_processed, target_col="Diagnosis")

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("y_test distribution:")
print(y_test.value_counts(normalize=True).round(3))

2026-07-03 16:45:51,486 | data_cleaning | INFO | Cleaning started. Input shape: (15000, 22)
2026-07-03 16:45:51,518 | data_cleaning | INFO | Filled 'Symptoms' with mode (16.7% missing)
2026-07-03 16:45:51,608 | data_cleaning | INFO | Removed 0 duplicate rows
2026-07-03 16:45:51,619 | data_cleaning | INFO | Skipped 'Hypertension' for outliers (2 unique values)
2026-07-03 16:45:51,619 | data_cleaning | INFO | Skipped 'Heart Disease' for outliers (2 unique values)
2026-07-03 16:45:51,650 | data_cleaning | INFO | Skipped 'Stroke History' for outliers (2 unique values)
2026-07-03 16:45:51,654 | data_cleaning | INFO | Cleaning finished. Output shape: (15000, 22)
2026-07-03 16:45:51,662 | feature_engineering | INFO | Feature processing started. Input shape: (15000, 22)
2026-07-03 16:45:57,534 | feature_engineering | INFO | Feature engineering: dropped 5 columns
2026-07-03 16:45:57,635 | feature_engineering | INFO | Encoding: 3 binary, 6 one-hot
2026-07-03 16:45:57,680 | feature_engineering | 

X_train: (12000, 34)
X_test : (3000, 34)
y_train distribution:
Diagnosis
0    0.502
1    0.498
Name: proportion, dtype: float64
y_test distribution:
Diagnosis
0    0.502
1    0.498
Name: proportion, dtype: float64


In [3]:
def select_by_mutual_info(X_train, y_train, min_mi=0.001):
    

    mi_scores = mutual_info_classif(
        X_train,
        y_train,
        random_state=42
    )

    mi_results = pd.DataFrame({
        "Feature": X_train.columns,
        "MI_Score": mi_scores
    })

    mi_results = mi_results.sort_values(
        by="MI_Score",
        ascending=False
    ).reset_index(drop=True)

    selected_features = mi_results[
        mi_results["MI_Score"] >= min_mi
    ]["Feature"].tolist()

    removed_features = mi_results[
        mi_results["MI_Score"] < min_mi
    ]["Feature"].tolist()

    return selected_features, mi_results, removed_features

In [4]:
def select_features(df, target, selected_cols=None):
    
    df_new = df.copy()
    
    if selected_cols is None:
        report = {"start_features": df_new.shape[1] - 1}
        df_new, mi_series, low_mi_dropped = select_by_mutual_info(df_new, target)
        report["low_mi_dropped"] = low_mi_dropped
        report["mi_scores"] = mi_series.round(4).to_dict()
        selected_cols = [c for c in df_new.columns if c != target]
        report["final_features"] = len(selected_cols)
        print(f"Feature selection (fit): {report['start_features']} -> {report['final_features']} features")
        return df_new, selected_cols, report
    
    else:
        keep = [c for c in selected_cols if c in df_new.columns]
        if target in df_new.columns:
            keep = keep + [target]
        df_new = df_new[keep]
        print(f"Feature selection (transform): kept {len(selected_cols)} features")
        return df_new, selected_cols, {}

In [10]:
def detect_class_imbalance(y):
    

    class_counts = y.value_counts().sort_index()

    imbalance_report = pd.DataFrame({
        "Class": class_counts.index,
        "Count": class_counts.values,
        "Percentage": (class_counts.values / len(y) * 100).round(2)
    })

    print("=" * 50)
    print("Class Distribution")
    print("=" * 50)
    print(imbalance_report)

    majority = class_counts.max()
    minority = class_counts.min()

    imbalance_ratio = round(majority / minority, 2)

    print(f"\nImbalance Ratio : {imbalance_ratio}:1")

    if imbalance_ratio > 1.5:
        print("Dataset is Imbalanced.")
    else:
        print("Dataset is Balanced.")

    return imbalance_report, imbalance_ratio

In [11]:
def compute_balanced_weights(y):
  

    classes = np.unique(y)

    weights = compute_class_weight(
        class_weight="balanced",
        classes=classes,
        y=y
    )

    class_weights = dict(zip(classes, weights))

    print("\nComputed Class Weights")
    print("-" * 30)

    for cls, weight in class_weights.items():
        print(f"Class {cls}: {weight:.4f}")

    return class_weights

In [12]:
imbalance_report, imbalance_ratio = detect_class_imbalance(y_train)

class_weights = compute_balanced_weights(y_train)

Class Distribution
   Class  Count  Percentage
0      0   6026       50.22
1      1   5974       49.78

Imbalance Ratio : 1.01:1
Dataset is Balanced.

Computed Class Weights
------------------------------
Class 0: 0.9957
Class 1: 1.0044


In [13]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    class_weight=class_weights,
    random_state=42,
    max_iter=1000
)

In [14]:
class_weight="balanced"